# Topic Modeling with BERTopic

Advancing AI Anthropology through computational approaches to qualitative research.

[Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

<br>

---

<br>

## What This Notebook Does

This notebook discovers topics in text collections using BERTopic, a transformer-based topic modeling framework. Upload documents (interview transcripts, field notes, articles, or any text corpus), and the notebook clusters them by semantic similarity, identifies topics, and produces interactive visualizations showing how topics relate to each other.

Unlike traditional topic modeling (LDA), BERTopic uses transformer embeddings to understand meaning, not just word frequency. This produces more coherent and interpretable topics, especially for qualitative research data where context matters.

## Key Features

1. **Transformer-Based**: Uses sentence embeddings for semantically meaningful topic clusters
2. **Zero-Shot Mode**: Optionally provide expected topics and let BERTopic find them in your data
3. **Multi-Format Input**: Upload CSV, TXT, or DOCX files, or paste text directly
4. **Interactive Visualizations**: Topic maps, hierarchy trees, bar charts, and similarity heatmaps
5. **Topic Search**: Find topics similar to any keyword or phrase
6. **Configurable Parameters**: Control topic count, minimum topic size, and n-gram range
7. **Structured Export**: Download topic assignments, topic summaries, and representative documents

## Workflow

1. **Upload**: Load your text corpus from files or paste directly
2. **Configure**: Set parameters and optionally define expected topics (zero-shot mode)
3. **Model**: Run BERTopic to discover topics in your data
4. **Explore**: Review visualizations and topic details
5. **Export**: Download topic assignments and summaries for further analysis

## Applications

This notebook supports research that involves discovering thematic structure in text collections. It complements the Coding and Thematic Analysis notebook (which applies predefined codes) by offering unsupervised topic discovery. Useful for exploratory analysis of interview corpora, literature collections, media coverage, field notes, and any text dataset where you want to identify emergent themes.

## Methodological Positioning

This notebook represents a **computational anthropology** approach. BERTopic discovers statistical patterns in text, not cultural meaning. The topics it identifies are starting points for interpretation, not conclusions. The researcher's ethnographic knowledge remains essential for evaluating whether discovered topics are analytically meaningful.

**Important**: Topic modeling is exploratory. Results depend on corpus composition, preprocessing choices, and parameter settings. Run with different configurations and compare results before drawing conclusions.

## Target Audience

Designed for anthropologists and qualitative researchers who want to explore thematic structure in text collections before or alongside manual coding.

## Technical Approach

The notebook uses **BERTopic** with sentence-transformer embeddings, UMAP dimensionality reduction, and HDBSCAN clustering. Topic labels are generated using KeyBERTInspired representation (no API key required). Zero-shot mode uses cosine similarity to match documents to predefined topics. All processing runs in the notebook.

## License & Attribution

This notebook is released under the **CC BY-NC 4.0** license. You may adapt and share it for non-commercial purposes with proper attribution.

## Citation

If you use this toolkit in your academic research, please cite:

> Artz, Matt. 2025. AI Anthropology Toolkit. Software. Zenodo. https://doi.org/10.5281/zenodo.16728812

## References

Artz, Matt. 2023. From Machine Learning to Machine Knowing: A Digital Anthropology Approach for the Machine Interpretation of Cultures. UNESCO. https://unesdoc.unesco.org/ark:/48223/pf0000384902.

Artz, Matt. 2023. "Ten Predictions for AI and the Future of Anthropology." Anthropology News, May 8. https://doi.org/10.1111/AN.1605.

Artz, Matt. 2026. "Artificial Intelligence: The AI Anthropology Lifecycle (of, by, for AI)." In Practicing Digital Ethnography, edited by Devin Proctor. Routledge. https://doi.org/10.4324/9781032672663-29.

Artz, Matt. 2026. "Multi-Agent Ethnography: Post-Conventional Anthropological Practice Through Human-AI Collaboration." Human Organization. https://doi.org/10.1080/00664677.2026.2614501.

Artz, Matt. Forthcoming. "AI Anthropology: The Future of Applied Anthropological Practice." In Routledge Handbook of Applied Anthropology, edited by Christina Wasson, Edward B. Liebow, Karine L. Narahara, Ndukuyakhe Ndlovu, and Alaka Wali. New York: Routledge.

Grootendorst, Maarten. 2022. "BERTopic: Neural Topic Modeling with a Class-Based TF-IDF Procedure." arXiv preprint arXiv:2203.05794.

## Setup and Installation

Install required packages and import libraries. This cell installs BERTopic and its dependencies. In Google Colab, a GPU runtime is recommended for faster processing (Runtime > Change runtime type > T4 GPU).

In [ ]:
# Install required packages
!pip install bertopic pandas openpyxl ipywidgets python-docx nltk -q

import re
import unicodedata
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

import logging
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

STOP_WORDS = set(stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()


def preprocess_text(text):
    """Preprocess text for topic modeling."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [LEMMATIZER.lemmatize(w) for w in tokens if w not in STOP_WORDS and len(w) > 2]
    return ' '.join(tokens)


def normalize_text(text):
    """Normalize unicode characters."""
    if not isinstance(text, str): return text
    text = unicodedata.normalize('NFKC', text)
    for old, new in [('\u2011','-'),('\u2013','-'),('\u2014','-'),('\u2018',"'"),('\u2019',"'"),('\u201c','"'),('\u201d','"'),('\u2026','...')]:
        text = text.replace(old, new)
    return text


def make_slug(query):
    """Create a clean filename slug."""
    return re.sub(r'[^a-zA-Z0-9]+', '_', query).strip('_')[:30] or 'topics'


def style_excel(filepath):
    """Apply branded styling to an Excel file."""
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    wb = load_workbook(filepath)
    hf = PatternFill(start_color='274C77', end_color='274C77', fill_type='solid')
    hfont = Font(bold=True, color='FFFFFF')
    tb = Border(left=Side(style='thin',color='E7ECEF'), right=Side(style='thin',color='E7ECEF'),
               top=Side(style='thin',color='E7ECEF'), bottom=Side(style='thin',color='E7ECEF'))
    for ws in wb.worksheets:
        for cell in ws[1]: cell.fill, cell.font, cell.alignment, cell.border = hf, hfont, Alignment(horizontal='center', vertical='center', wrap_text=True), tb
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = min(max(max(len(str(c.value or '')) for c in col)+2, 10), 60)
        ws.freeze_panes = 'A2'
        for row in ws.iter_rows(min_row=2):
            for cell in row: cell.alignment, cell.border = Alignment(vertical='top', wrap_text=True), tb
    wb.save(filepath)


print("\u2713 All packages installed and libraries loaded successfully")
print(f"\u2713 Environment: {"Google Colab" if IN_COLAB else "Local Jupyter"}")
print("\U0001f4ca Ready to configure topic modeling!")

## Data Input

Upload your text data or paste it directly. Supports CSV files (specify the text column), TXT files (one document per line or per file), and DOCX files.

In [ ]:
# Data Input Interface

MIN_CHUNK_WORDS = 80  # Merge short paragraphs until chunks reach this minimum

style = {'description_width': '120px'}
layout = widgets.Layout(width='500px')

input_html = '<div style="background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;">'
input_html += '<h2 style="color: #274C77; margin-top: 0;">\U0001f4ca Topic Modeling with BERTopic</h2>'
input_html += '<p><strong>Welcome!</strong> This notebook discovers topics in your text data using transformer-based clustering.</p>'
input_html += '<h3 style="color: #274C77;">\U0001f4d6 How to Use:</h3>'
input_html += '<ol>'
input_html += '<li><strong>Upload:</strong> Add your text data below (CSV, TXT, DOCX, or PDF)</li>'
input_html += '<li><strong>Configure:</strong> Set parameters in the next cell</li>'
input_html += '<li><strong>Run:</strong> Execute the model to discover topics</li>'
input_html += '<li><strong>Explore:</strong> Review visualizations and export results</li>'
input_html += '</ol>'
input_html += '</div>'

input_instructions = widgets.HTML(value=input_html)

input_method = widgets.ToggleButtons(
    options=[('Upload Files', 'upload'), ('Paste Text', 'paste')],
    value='upload',
    style={'button_width': '150px'}
)

# CSV column setting (shown in upload mode)
csv_col = widgets.Text(value='text', description='Text column:', placeholder='Column name containing text (for CSV)', layout=layout, style=style)
csv_help = widgets.HTML('<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">For CSV files, specify which column contains the text to analyze.</p>')

# Paste widget
paste_area = widgets.Textarea(value='', placeholder='Paste documents here, one per line (or separated by blank lines)', layout=widgets.Layout(width='100%', height='200px'))

# Storage
documents = []
doc_labels = []
upload_out = widgets.Output()

upload_box = widgets.VBox([csv_col, csv_help])
paste_box = widgets.VBox([paste_area])
paste_box.layout.display = 'none'

def on_method_change(change):
    if change['new'] == 'upload':
        upload_box.layout.display = ''
        paste_box.layout.display = 'none'
    else:
        upload_box.layout.display = 'none'
        paste_box.layout.display = ''

input_method.observe(on_method_change, names='value')


def merge_short_paragraphs(paragraphs, min_words=MIN_CHUNK_WORDS):
    """Merge consecutive short paragraphs into chunks of at least min_words.

    Prevents BERTopic from getting tiny fragments like headings and captions
    as standalone documents. Adjacent paragraphs are combined until the chunk
    reaches the minimum word count.
    """
    chunks = []
    current = []
    current_words = 0

    for para in paragraphs:
        words = len(para.split())
        current.append(para)
        current_words += words

        if current_words >= min_words:
            chunks.append(' '.join(current))
            current = []
            current_words = 0

    # Append remainder
    if current:
        if chunks:
            # Merge trailing short fragment into the last chunk
            chunks[-1] += ' ' + ' '.join(current)
        else:
            chunks.append(' '.join(current))

    return chunks


load_btn = widgets.Button(description='Load Documents', style={'button_color': '#6096BA'}, layout=widgets.Layout(width='200px', margin='20px 0'))

def on_load_clicked(_):
    global documents, doc_labels
    upload_out.clear_output()
    documents = []
    doc_labels = []

    with upload_out:
        if input_method.value == 'paste':
            raw = paste_area.value.strip()
            if not raw:
                print("\u26a0\ufe0f Please paste some text.")
                return
            # Split by blank lines or single lines
            docs = [d.strip() for d in re.split(r'\n\s*\n', raw) if d.strip()]
            if len(docs) == 1:
                docs = [d.strip() for d in raw.split('\n') if d.strip()]
            docs = merge_short_paragraphs(docs)
            documents = docs
            doc_labels = [f'doc_{i+1}' for i in range(len(docs))]
            print(f"\u2713 Loaded {len(documents)} text segments from pasted text")

        elif input_method.value == 'upload':
            if IN_COLAB:
                from google.colab import files as colab_files
                uploaded = colab_files.upload()
                if not uploaded:
                    print("\u26a0\ufe0f No files uploaded.")
                    return
                for fname, content in uploaded.items():
                    if fname.endswith('.csv'):
                        import io
                        df = pd.read_csv(io.BytesIO(content))
                        col = csv_col.value.strip()
                        if col not in df.columns:
                            print(f"\u26a0\ufe0f Column '{col}' not found. Available: {list(df.columns)}")
                            return
                        docs = df[col].dropna().astype(str).tolist()
                        documents.extend(docs)
                        doc_labels.extend([f'{fname}_{i+1}' for i in range(len(docs))])
                    elif fname.endswith('.txt'):
                        text = content.decode('utf-8', errors='ignore')
                        paras = [d.strip() for d in text.split('\n') if d.strip()]
                        docs = merge_short_paragraphs(paras)
                        documents.extend(docs)
                        doc_labels.extend([f'{fname}_{i+1}' for i in range(len(docs))])
                    elif fname.endswith('.docx'):
                        import docx, io
                        doc = docx.Document(io.BytesIO(content))
                        paras = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
                        docs = merge_short_paragraphs(paras)
                        documents.extend(docs)
                        doc_labels.extend([f'{fname}_{i+1}' for i in range(len(docs))])
                    elif fname.endswith('.pdf'):
                        import io
                        try:
                            from pypdf import PdfReader
                        except ImportError:
                            from PyPDF2 import PdfReader
                        reader = PdfReader(io.BytesIO(content))
                        text = '\n'.join(page.extract_text() or '' for page in reader.pages)
                        paras = [d.strip() for d in text.split('\n') if d.strip()]
                        docs = merge_short_paragraphs(paras)
                        documents.extend(docs)
                        doc_labels.extend([f'{fname}_{i+1}' for i in range(len(docs))])
                    else:
                        print(f"\u26a0\ufe0f Unsupported format: {fname}")
                        continue
                print(f"\u2713 Loaded {len(documents)} text segments from {len(uploaded)} file(s)")
            else:
                print("\u26a0\ufe0f File upload widget works differently locally. Use Paste Text mode, or load from CSV:")
                print("   Upload a CSV to your notebook directory and set the path below.")

        if documents:
            avg_len = np.mean([len(d.split()) for d in documents])
            print(f"   Average segment length: {avg_len:.0f} words")
            if len(documents) < 20:
                print(f"   \u26a0\ufe0f Small corpus ({len(documents)} segments). BERTopic works best with 50+ segments.")

load_btn.on_click(on_load_clicked)

display(widgets.VBox([
    input_instructions,
    input_method,
    upload_box,
    paste_box,
    load_btn,
    upload_out,
]))

## Model Configuration

Configure BERTopic parameters. In standard mode, BERTopic discovers topics automatically. In zero-shot mode, you provide a list of expected topics and BERTopic finds documents matching them.

In [ ]:
# Model Configuration

config_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\u2699\ufe0f Parameters</h3>')

mode_toggle = widgets.ToggleButtons(
    options=[('Standard (auto-discover)', 'standard'), ('Zero-Shot (guided)', 'zeroshot')],
    value='standard',
    style={'button_width': '200px'}
)

mode_help = widgets.HTML(
    '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
    '<strong>Standard:</strong> BERTopic discovers topics automatically from your data.<br>'
    '<strong>Zero-Shot:</strong> Provide expected topics. Documents matching them are assigned; the rest are clustered normally.</p>'
)

min_topic_input = widgets.IntSlider(value=10, min=3, max=50, step=1, description='Min topic size:', style=style)
min_topic_help = widgets.HTML('<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">Minimum number of documents to form a topic. Lower = more topics, higher = fewer broader topics.</p>')

nr_topics_input = widgets.Dropdown(
    options=[('Auto', 'auto'), ('10', 10), ('20', 20), ('30', 30), ('50', 50), ('No reduction', None)],
    value='auto',
    description='Nr topics:',
    style=style
)

ngram_input = widgets.Dropdown(
    options=[('Unigrams (1)', 1), ('Bigrams (1-2)', 2), ('Trigrams (1-3)', 3)],
    value=2,
    description='N-gram range:',
    style=style
)

preprocess_chk = widgets.Checkbox(value=True, description='Preprocess text (stopword removal, lemmatization)', indent=False, style={'description_width': 'auto'})

# Zero-shot config
zeroshot_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f3af Zero-Shot Topics</h3>')
zeroshot_input = widgets.Textarea(
    value='',
    placeholder='Enter expected topics, one per line (e.g., "Healthcare access", "Community resilience")',
    layout=widgets.Layout(width='500px', height='100px')
)
zeroshot_sim = widgets.FloatSlider(value=0.85, min=0.5, max=0.99, step=0.01, description='Min similarity:', style=style)
zeroshot_help = widgets.HTML('<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">Documents below this similarity threshold will be clustered normally instead of assigned to a zero-shot topic.</p>')

zeroshot_box = widgets.VBox([zeroshot_header, zeroshot_input, zeroshot_sim, zeroshot_help])
zeroshot_box.layout.display = 'none'

def on_mode_toggle(change):
    zeroshot_box.layout.display = '' if change['new'] == 'zeroshot' else 'none'
mode_toggle.observe(on_mode_toggle, names='value')

display(widgets.VBox([
    config_header,
    mode_toggle, mode_help,
    min_topic_input, min_topic_help,
    nr_topics_input, ngram_input,
    preprocess_chk,
    zeroshot_box,
]))

## Run Topic Model

Execute BERTopic on your loaded documents. This may take a few minutes depending on corpus size. Progress updates appear below.

In [ ]:
# Run BERTopic

topic_model = None
topics = None
probs = None
docs_processed = None

run_btn = widgets.Button(description='Run Topic Model', style={'button_color': '#6096BA'}, layout=widgets.Layout(width='200px', margin='10px 0'))
progress = widgets.HTML(value='')
run_out = widgets.Output()

def on_run_clicked(_):
    global topic_model, topics, probs, docs_processed
    run_out.clear_output()
    progress.value = ''

    if not documents:
        with run_out:
            print("\u26a0\ufe0f No documents loaded. Run the Data Input cell first.")
        return

    with run_out:
        print(f"\U0001f4ca Running BERTopic on {len(documents)} documents...")
        print()

    progress.value = '<span style="color: #6096BA;">Preprocessing text...</span>'

    # Preprocess
    if preprocess_chk.value:
        docs_processed = [preprocess_text(d) for d in documents]
    else:
        docs_processed = [str(d) for d in documents]

    # Remove empty docs
    valid = [(d, l, orig) for d, l, orig in zip(docs_processed, doc_labels, documents) if d.strip()]
    if not valid:
        with run_out:
            print("\u26a0\ufe0f All documents were empty after preprocessing.")
        progress.value = ''
        return

    docs_clean, labels_clean, originals = zip(*valid)
    docs_clean = list(docs_clean)

    with run_out:
        print(f"   {len(docs_clean)} documents after preprocessing")

    progress.value = '<span style="color: #6096BA;">Building topic model (this may take a few minutes)...</span>'

    try:
        representation_model = KeyBERTInspired()

        model_kwargs = {
            'min_topic_size': min_topic_input.value,
            'representation_model': representation_model,
            'n_gram_range': (1, ngram_input.value),
            'verbose': False,
        }

        # Zero-shot config
        if mode_toggle.value == 'zeroshot':
            zt = [t.strip() for t in zeroshot_input.value.strip().split('\n') if t.strip()]
            if zt:
                model_kwargs['zeroshot_topic_list'] = zt
                model_kwargs['zeroshot_min_similarity'] = zeroshot_sim.value
                with run_out:
                    print(f"   Zero-shot topics: {zt}")

        topic_model = BERTopic(**model_kwargs)
        topics, probs = topic_model.fit_transform(docs_clean)

        # Reduce topics if requested
        nr = nr_topics_input.value
        if nr is not None and nr != 'auto':
            topic_model.reduce_topics(docs_clean, nr_topics=int(nr))
            topics = topic_model.topics_

        progress.value = ''

        with run_out:
            topic_info = topic_model.get_topic_info()
            n_topics = len(topic_info[topic_info['Topic'] != -1])
            n_outliers = len([t for t in topics if t == -1])
            print(f"\u2713 Found {n_topics} topics ({n_outliers} outlier documents)")
            print()
            print("\U0001f4cb Topic Summary:")
            display(topic_info.head(20))

    except Exception as e:
        progress.value = ''
        with run_out:
            print(f"\u274c Error: {type(e).__name__}: {e}")

run_btn.on_click(on_run_clicked)
display(widgets.VBox([run_btn, progress, run_out]))

## Visualizations

Explore the discovered topics through interactive visualizations. Each visualization offers a different perspective on the topic structure.

In [ ]:
# Topic Visualizations

if topic_model is None:
    print("\u26a0\ufe0f Run the topic model first.")
else:
    print("\U0001f4ca Topic Visualizations")
    print("=" * 40)

    # Topic overview (intertopic distance map)
    print("\n1. Topic Map (intertopic distance):")
    try:
        fig = topic_model.visualize_topics()
        fig.show()
    except Exception as e:
        print(f"   Could not render topic map: {e}")

    # Bar chart of top terms per topic
    print("\n2. Top Terms per Topic:")
    try:
        fig = topic_model.visualize_barchart(top_n_topics=min(10, len(topic_model.get_topic_info()) - 1))
        fig.show()
    except Exception as e:
        print(f"   Could not render bar chart: {e}")

    # Topic hierarchy
    print("\n3. Topic Hierarchy:")
    try:
        fig = topic_model.visualize_hierarchy()
        fig.show()
    except Exception as e:
        print(f"   Could not render hierarchy: {e}")

    # Topic similarity heatmap
    print("\n4. Topic Similarity Heatmap:")
    try:
        fig = topic_model.visualize_heatmap()
        fig.show()
    except Exception as e:
        print(f"   Could not render heatmap: {e}")

## Topic Search

Search for topics similar to any keyword or phrase. Useful for finding which discovered topics relate to concepts you care about.

In [ ]:
# Topic Search

if topic_model is None:
    print("\u26a0\ufe0f Run the topic model first.")
else:
    search_input = widgets.Text(value='', placeholder='Enter a keyword or phrase', description='Search:', layout=layout, style=style)
    search_btn = widgets.Button(description='Find Topics', style={'button_color': '#6096BA'}, layout=widgets.Layout(width='150px', margin='10px 0'))
    search_out = widgets.Output()

    def on_search(_):
        search_out.clear_output()
        query = search_input.value.strip()
        if not query:
            return
        with search_out:
            similar_topics, similarity = topic_model.find_topics(query, top_n=5)
            print(f"\U0001f50d Topics similar to \"{query}\":\n")
            for t, s in zip(similar_topics, similarity):
                topic_words = topic_model.get_topic(t)
                if topic_words:
                    words = ', '.join([w for w, _ in topic_words[:5]])
                    print(f"  Topic {t} (similarity: {s:.3f}): {words}")

    search_btn.on_click(on_search)
    display(widgets.VBox([search_input, search_btn, search_out]))

## Export Results

Export topic assignments and summaries as CSV or Excel. The export includes document-level topic assignments and a topic summary sheet.

In [ ]:
# Export Results

if topic_model is None:
    print("\u26a0\ufe0f Run the topic model first.")
else:
    export_format = widgets.Dropdown(options=[('CSV (.csv)', 'csv'), ('Excel (.xlsx)', 'xlsx')], value='csv', description='Format:', style=style, layout=layout)
    export_btn = widgets.Button(description='Export Results', style={'button_color': '#6096BA'}, layout=widgets.Layout(width='200px', margin='10px 0'))
    export_out = widgets.Output()

    def on_export(_):
        export_out.clear_output()
        with export_out:
            try:
                timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                base = f'bertopic_results_{timestamp}'

                # Document-level assignments
                doc_df = pd.DataFrame({
                    'document': [documents[i] if i < len(documents) else '' for i in range(len(topics))],
                    'topic': topics,
                    'topic_label': [topic_model.get_topic_info().set_index('Topic').loc[t, 'Name'] if t in topic_model.get_topic_info()['Topic'].values else 'Outlier' for t in topics],
                })

                # Topic summary
                topic_info = topic_model.get_topic_info()

                fmt = export_format.value
                if fmt == 'xlsx':
                    filepath = f'{base}.xlsx'
                    with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
                        doc_df.to_excel(writer, sheet_name='Document Topics', index=False)
                        topic_info.to_excel(writer, sheet_name='Topic Summary', index=False)
                    style_excel(filepath)
                else:
                    filepath = f'{base}_documents.csv'
                    doc_df.to_csv(filepath, index=False)
                    summary_path = f'{base}_topics.csv'
                    topic_info.to_csv(summary_path, index=False)

                print(f"\u2713 Saved: {filepath} ({len(doc_df)} documents, {len(topic_info)} topics)")

                if IN_COLAB:
                    colab_files.download(filepath)
                    if fmt == 'csv':
                        colab_files.download(summary_path)

            except Exception as e:
                print(f"\u274c Export error: {type(e).__name__}: {e}")

    export_btn.on_click(on_export)
    display(widgets.VBox([export_format, export_btn, export_out]))